In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from datetime import date, datetime, timedelta
from pyspark.sql.window import Window
import re

catalog = "case"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

print(f"Schema {silver_schema} garantido. Iniciando transformações...")

In [0]:
df_bronze_clientes = spark.table(f"{bronze_schema}.tb_clientes")
df_silver_clientes = df_bronze_clientes


# display(df_silver_clientes)

### Padronização dos nomes

Esta etapa limpa e padroniza a coluna `nome`, para que nomes iguais escritos de formas diferentes fiquem idênticos e o texto fique pronto para exibição.

**Etapas aplicadas**

1. **Remoção de caracteres inválidos:** mantém apenas letras (inclusive acentuadas), espaços, apóstrofos (`'`) e hífens (`-`). Números, pontuação e símbolos são removidos.
   - O caractere `�` é mantido de propósito: ele indica uma letra corrompida por problema de encoding na origem e permite identificar esses registros depois.
2. **Espaços:** espaços duplos ou repetidos viram um único espaço, e os espaços no início e no fim são removidos.
3. **Maiúsculas e minúsculas:** a primeira letra de cada palavra fica maiúscula e as demais minúsculas (`initcap`).
4. **Preposições:** `da`, `de`, `do`, `das`, `dos` e `e` ficam em minúsculo quando aparecem no meio do nome, como é o padrão em nomes brasileiros.
5. **Nomes vazios:** se o nome ficar vazio depois da limpeza, ele é substituído por `null`.

**Exemplos**

| Antes | Depois |
|---|---|
| `  MARIA   DA SILVA ` | `Maria da Silva` |
| `joão dos santos` | `João dos Santos` |
| `Ana Paula123` | `Ana Paula` |
| `jos� de souza` | `Jos� de Souza` |
| `123` | `null` |

**Limitações conhecidas**

- O `initcap` não reconhece nomes com maiúscula no meio, como `O'Connor` (vira `O'connor`) ou `McDonald` (vira `Mcdonald`).
- Preposições no início do nome não são alteradas (ex.: `Da Silva Ana` permanece igual).


In [0]:
#Padronizando os Nomes

nome = F.col("nome")
nome = F.regexp_replace(nome, r"[^\p{L}\p{M}\uFFFD '\-]", "") # mantém só letras (inclusive acentuadas), espaço, apóstrofo e hífen
nome = F.regexp_replace(nome, r"\s+", " ") # espaços duplos viram um
nome = F.initcap(F.trim(nome))  # "maria DA silva" -> "Maria Da Silva"

# preposições em minúsculo: "Maria Da Silva" -> "Maria da Silva"
for p in ["Da", "De", "Do", "Das", "Dos", "E"]:
    nome = F.regexp_replace(nome, f"(?<= ){p}(?= )", p.lower())

df_silver_clientes = df_silver_clientes.withColumn(
    "nome",
    F.when(nome == "", None).otherwise(nome) # nome que ficou vazio após a limpeza vira null
)

# display(df_silver_clientes.select("nome"))


### Tratamento de e-mail

Esta etapa remove os espaços dos e-mails, identifica os que estão fora do padrão e corrige os que ficaram sem o `@` antes de um domínio conhecido.

---

#### 1. Remoção de espaços

Um e-mail válido nunca tem espaços. Por isso, todos são removidos, inclusive:
- espaços comuns, tabs e quebras de linha (`\s`);
- o espaço não separável (`\u00A0`), invisível e comum em dados copiados do Excel ou de páginas web.

Exemplo: `maria @gmail.com` → `maria@gmail.com`

---

#### 2. Diagnóstico: e-mails fora do padrão

Cada e-mail é comparado com a expressão regular abaixo, depois de convertido para minúsculo:

`^[a-z0-9._%+\-]+@[a-z0-9\-]+(\.[a-z0-9\-]+)*\.[a-z]{2,}$`

| Trecho | Significado |
|---|---|
| `[a-z0-9._%+\-]+` | usuário: letras, números e os símbolos `. _ % + -` |
| `@` | exatamente um `@` |
| `[a-z0-9\-]+` | nome do domínio (ex.: `gmail`) |
| `(\.[a-z0-9\-]+)*` | subdomínios opcionais (ex.: `.com` em `.com.br`) |
| `\.[a-z]{2,}$` | final com ponto e pelo menos 2 letras (ex.: `.br`, `.com`) |

A tabela de resultado mostra cada e-mail fora do padrão e quantas vezes ele aparece, do mais frequente para o menos frequente. Valores muito repetidos costumam ser preenchimentos falsos (ex.: `naotem`, `teste@teste`). E-mails nulos não entram nessa contagem.

---

#### 3. Correção: inclusão do `@` faltante

Corrige e-mails em que o usuário esqueceu o `@` antes de um domínio conhecido.

**Como funciona**

1. É montada uma lista de domínios conhecidos (`gmail.com`, `hotmail.com`, `uol.com.br` etc.).
2. Um e-mail só é corrigido se **não tiver `@`** e **terminar com um domínio da lista**. Esses registros são marcados na coluna auxiliar `faltou_arroba`.
3. A expressão regular separa o texto em duas partes, usuário e domínio, e remonta como `usuário@domínio`:

`^(.+?)[.,]?(gmail\.com|hotmail\.com|...)$`

| Trecho | Significado |
|---|---|
| `(.+?)` | usuário. O `?` faz o Spark testar primeiro o menor usuário possível, garantindo que o domínio mais longo seja reconhecido (ex.: `hotmail.com.br` em vez de `hotmail.com`) |
| `[.,]?` | absorve um ponto ou vírgula digitado no lugar do `@` |
| `(gmail\.com\|...)$` | domínio da lista, obrigatoriamente no final do texto |

O `re.escape` transforma o `.` dos domínios em `\.`, para que ele seja lido como ponto literal e não como "qualquer caractere".

**Exemplos**

| Antes | Depois |
|---|---|
| `mariagmail.com` | `maria@gmail.com` |
| `maria.gmail.com` | `maria@gmail.com` |
| `joaohotmail.com.br` | `joao@hotmail.com.br` |
| `ana.silva,outlook.com` | `ana.silva@outlook.com` |
| `maria` | `maria` (sem domínio conhecido, não é alterado) |

**Conferência:** a célula exibe a quantidade de e-mails corrigidos e uma amostra com o valor original ao lado do corrigido. As colunas auxiliares (`email_original` e `faltou_arroba`) são removidas no final.

---

**Limitações**

- A validação confere apenas o **formato**. Um e-mail como `maria123@gmail.com` é considerado válido mesmo que o endereço não exista.
- Apenas os domínios da lista são corrigidos. Para cobrir mais casos, basta incluir novos domínios. Domínios genéricos e curtos (como `mail.com`) devem ser evitados, pois podem casar com o final de outros domínios e gerar correções erradas.

In [0]:

#Tratamento de email

REGEX_EMAIL = r"^[a-z0-9._%+\-]+@[a-z0-9\-]+(\.[a-z0-9\-]+)*\.[a-z]{2,}$"

#Retira os espacos " " dos emails para nao ocorrer problema
df_silver_clientes = df_silver_clientes.withColumn(
    "email",
    F.regexp_replace(F.col("email"), r"[\s\u00A0]+", "")
)
# Email defeituosos(emails fora do padrao de email) para analise
df_check = df_silver_clientes.withColumn("email_limpo", F.lower(F.trim(F.col("email"))))
cont_emails_defeituosos =df_check.filter(F.col("email").isNotNull() & ~F.col("email_limpo").rlike(REGEX_EMAIL)).count()

#Mostra os emails defeituosos
# display(
#     df_check.filter(F.col("email").isNotNull() & ~F.col("email_limpo").rlike(REGEX_EMAIL))
#     .groupBy("email").count()
#     .orderBy(F.desc("count"))
# )

print(f"Quantidade de Emails Defeituosos:{cont_emails_defeituosos}")

In [0]:
#Adiciona @ para os emails que esqueceram

DOMINIOS = [
    "gmail.com", "hotmail.com", "hotmail.com.br", "outlook.com", "outlook.com.br",
    "yahoo.com", "yahoo.com.br", "icloud.com", "live.com", "msn.com",
    "uol.com.br", "bol.com.br", "terra.com.br", "ig.com.br", "globo.com",
]

padrao_dominios = "|".join(re.escape(d) for d in DOMINIOS)
REGEX_SEM_ARROBA = rf"^(.+?)[.,]?({padrao_dominios})$"

# Guarda o valor original, para conferir depois
df = df_silver_clientes.withColumn("email_original", F.col("email"))

# Limpeza básica: minúsculo
df = df.withColumn("email", F.lower(F.col("email")))

# Marca quem esqueceu o "@" antes de um domínio conhecido
df = df.withColumn(
    "faltou_arroba",
    F.coalesce(
        ~F.col("email").contains("@") & F.col("email").rlike(REGEX_SEM_ARROBA),
        F.lit(False)
    )
)

# Insere o "@"
df = df.withColumn(
    "email",
    F.when(
        F.col("faltou_arroba"),
        F.regexp_replace(F.col("email"), REGEX_SEM_ARROBA, "$1@$2")
    ).otherwise(F.col("email"))
)

# Conferência
print(f"E-mails corrigidos: {df.filter('faltou_arroba').count()}")
# display(df.filter("faltou_arroba").select("email_original", "email").limit(50))

df_silver_clientes = df.drop("email_original", "faltou_arroba")

#Remove linhas com email igual(Clientes podem ter nome iguais, mas nao emails)
df_silver_clientes = df_silver_clientes.dropDuplicates(["email"])

### Tratamento da coluna `regiao`

Esta etapa padroniza a coluna `regiao`, para que a mesma região escrita de formas diferentes seja reconhecida como um único valor.

**Etapas aplicadas**

1. **Minúsculo (`lower`):** todas as letras ficam minúsculas, unificando variações como `Sudeste`, `SUDESTE` e `sudeste`.
2. **Remoção de espaços nas pontas (`trim`):** remove os espaços no início e no fim do texto. Sem isso, `sudeste` e `sudeste ` seriam tratados como valores diferentes.

**Exemplos**

| Antes | Depois |
|---|---|
| `Sudeste` | `sudeste` |
| `SUDESTE ` | `sudeste` |
| ` Nordeste` | `nordeste` |

**Conferência:** a célula exibe a lista de regiões distintas após o tratamento, em ordem alfabética. Se ainda aparecerem variações da mesma região, elas precisam de tratamento adicional.

**Limitação:** acentos e erros de digitação não são tratados nesta etapa. Valores como `sudeste` e `sudéste`, ou `sul` e `sull`, continuariam aparecendo como regiões diferentes.

In [0]:
#Tratamento "regiao"

#Retira os espaços nos finais e deixa minusculo
df_silver_clientes = df_silver_clientes.withColumn("regiao", F.trim(F.lower(F.col("regiao"))))

# display(df_silver_clientes.select("regiao").distinct().orderBy("regiao"))

### Tratamento da coluna `idade`

Esta etapa classifica cada idade de acordo com a regra de negócio, para identificar valores ausentes ou fora do intervalo esperado.

**Regra de negócio**

| Parâmetro | Valor | Motivo |
|---|---|---|
| `IDADE_MIN` | 18 | clientes precisam ser maiores de idade |
| `IDADE_MAX` | 120 | idades acima disso são consideradas erro de cadastro |

**Classificação (coluna `status_idade`)**

| Status | Condição |
|---|---|
| `nulo_na_origem` | idade não informada |
| `abaixo_do_minimo` | idade menor que 18 (inclui valores como `0`, comuns quando o campo não foi preenchido) |
| `acima_do_maximo` | idade maior que 120 |
| `ok` | idade dentro do intervalo válido |

A ordem das condições importa: o Spark aplica a primeira que for verdadeira. Por isso, a checagem de nulo vem primeiro.

**Apuração:** a primeira tabela mostra quantos clientes há em cada status e o percentual que representam na base. Isso ajuda a dimensionar o problema: poucos casos indicam erros pontuais, enquanto um percentual alto pode indicar uma falha no sistema de origem.

**Conferência:** a segunda tabela mostra estatísticas da coluna `idade` (quantidade, mínimo, quartis, mediana, máximo e média), para verificar a distribuição dos valores.

**Por que a linha não é removida:** o cliente continua válido para outras análises (região, e-mail, data de cadastro). Para análises que dependem da idade, basta filtrar por `status_idade == "ok"`.

In [0]:
#Tratamento idade

# Regra de negócio
IDADE_MIN = 18
IDADE_MAX = 120

# 2. Classifica cada idade
df_silver_clientes = df_silver_clientes.withColumn(
    "status_idade",
    F.when(F.col("idade").isNull(), "nulo_na_origem")
     .when(F.col("idade") < IDADE_MIN, "abaixo_do_minimo")
     .when(F.col("idade") > IDADE_MAX, "acima_do_maximo")
     .otherwise("ok")
)

# Apuração
# display(
#     df_silver_clientes.groupBy("status_idade").count()
#     .withColumn("pct", F.round(F.col("count") / total * 100, 2))
#     .orderBy(F.desc("count"))
# )

# Conferência

### Tratamento da coluna `data_cadastro`

Esta etapa converte a coluna `data_cadastro` de texto (`string`) para data (`date`).

**Por que converter**

Enquanto a data é texto, o Spark não a reconhece como data, e isso causa problemas:
- **Ordenação errada:** texto é ordenado caractere por caractere, então `01/12/2025` viria antes de `14/10/2024`.
- **Formatos diferentes não se comparam:** `14/10/2024` e `2024-10-14` são o mesmo dia, mas como texto são valores distintos.
- **Sem cálculos:** funções como `year()`, `datediff()` e comparações com a data atual não funcionam corretamente.

---

#### 1. Verificação dos formatos

Cada dígito é substituído por `9`, de modo que datas com o mesmo formato caiam no mesmo grupo. Resultado encontrado na base:

| Formato | Quantidade | Exemplo |
|---|---|---|
| `99/99/9999` | 44.690 | `14/10/2024` |
| `9999-99-99` | 2.310 | `2026-07-07` |

O formato `99/99/9999` foi confirmado como **dia/mês/ano** (`dd/MM/yyyy`): o primeiro número chega a 31, o segundo nunca passa de 12, e não há nenhuma data no padrão americano (mês/dia/ano).

---

#### 2. Conversão para `date`

Cada valor é convertido de acordo com o seu formato:

| Padrão identificado | Formato usado na leitura |
|---|---|
| `^\d{2}/\d{2}/\d{4}$` (ex.: `14/10/2024`) | `dd/MM/yyyy` |
| `^\d{4}-\d{2}-\d{2}$` (ex.: `2024-10-14`) | `yyyy-MM-dd` |

- O formato é escolhido pela expressão regular (`rlike`), garantindo que nenhuma data seja lida no formato errado.
- O `try_to_timestamp` retorna `null` para datas impossíveis (ex.: `31/02/2024`) em vez de interromper a execução.
- O `.cast("date")` remove a parte de horário, mantendo apenas a data.

**Resultado:** as duas formas abaixo passam a ser o mesmo valor, do tipo `date`:

| Antes (texto) | Depois (date) |
|---|---|
| `14/10/2024` | `2024-10-14` |
| `2024-10-14` | `2024-10-14` |

**Observação:** o Databricks sempre exibe datas no formato `yyyy-MM-dd`. Esse é apenas o formato de exibição. O tipo `date` não armazena formato, e a exibição como `dd/MM/yyyy` pode ser feita nos relatórios e dashboards.

In [0]:
#Tratamento Data

#Verificacao dos formatos de  data
# display(
#     df_silver_clientes
#     .groupBy(F.regexp_replace(F.col("data_cadastro"), r"\d", "9").alias("formato"))
#     .agg(F.count("*").alias("qtd"), F.first("data_cadastro").alias("exemplo"))
#     .orderBy(F.desc("qtd"))
# )

#Convertendo de string para date
texto = F.trim(F.col("data_cadastro"))

df_silver_clientes = df_silver_clientes.withColumn(
    "data_cadastro",
    F.when(
        texto.rlike(r"^\d{2}/\d{2}/\d{4}$"),
        F.expr(f"try_to_timestamp(trim({"data_cadastro"}), 'dd/MM/yyyy')").cast("date")
    ).when(
        texto.rlike(r"^\d{4}-\d{2}-\d{2}$"),
        F.expr(f"try_to_timestamp(trim({"data_cadastro"}), 'yyyy-MM-dd')").cast("date")
    )
)

In [0]:
df_silver_clientes.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_clientes")


print(f"Tabela {silver_schema}.tb_chamados criada/atualizada com sucesso!")
print(f"Bronze (antes do dedup): {df_bronze_clientes.count():,} linhas")
print(f"Silver (após dedup e tratamento): {df_silver_clientes.count():,} linhas")